# LoRA vs QLoRA — a controlled ablation

The published result used **QLoRA** (4-bit NF4 base). At 1.5B, QLoRA was not required —
fp16 LoRA fits a 16 GB T4 too. This notebook runs the *identical* recipe with **only**
the base precision changed, and measures accuracy, wall clock, and peak VRAM.

It imports everything from the `text2sql_lora` package — the same code, the same metric,
the same three-system evaluation as the main run. The only change is one config flag.

**You only need to run the `lora_fp16` arm.** The QLoRA arm is the already-published
`artifacts/metrics.json`, which `collect_experiments.py` reads directly. Running the
QLoRA arm here too is optional — it reproduces the published number on the same hardware,
which makes the VRAM/wall-clock comparison exactly like-for-like.

## Sidebar settings

1. **Settings → Accelerator → GPU T4 x2** (only one GPU is used — pinned below).
2. **Settings → Internet → On.**
3. Private repo? Add the `GITHUB_TOKEN` secret (see `kaggle_lora_sft.ipynb`).

In [ ]:
# Pin ONE GPU before importing torch. Kaggle 'T4 x2' exposes two devices, and Trainer
# wraps the model in nn.DataParallel when it sees more than one -- which corrupts a
# bitsandbytes 4-bit layer and dies with an illegal memory access. Documented failure;
# do not remove.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/shashi05shankar/Text-to-SQL-LoRA-Supervised-Fine-Tuning.git"

import torch

print("torch", torch.__version__, "| visible GPUs:", torch.cuda.device_count(), "(must be 1)")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2.")
major, minor = torch.cuda.get_device_capability(0)
print(f"gpu: {torch.cuda.get_device_name(0)} | cc {major}.{minor} | "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ==================  CHOOSE THE ARM  ==================
# "lora_fp16"  -> non-quantized base (the new arm; run this)
# "qlora_nf4"  -> 4-bit NF4 base (reproduces the published run on this hardware; optional)
ARM = "lora_fp16"
# =====================================================
assert ARM in ("lora_fp16", "qlora_nf4")
LOAD_IN_4BIT = ARM == "qlora_nf4"
print(f"arm: {ARM}  |  load_in_4bit={LOAD_IN_4BIT}")

In [ ]:
# Install the project (git clone; token-authenticated if the repo is private).
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/text2sql-lora-sft")


def authenticated(url: str) -> str:
    token = globals().get("GITHUB_TOKEN")
    if not token or not url.startswith("https://github.com/"):
        return url
    return url.replace("https://", f"https://x-access-token:{token}@", 1)


def find_attached_repo():
    for candidate in Path("/kaggle/input").rglob("pyproject.toml"):
        try:
            if "text2sql-lora" in candidate.read_text(encoding="utf-8"):
                return candidate.parent
        except Exception:
            continue
    return None


if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

cloned = False
try:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", authenticated(REPO_URL), str(REPO_DIR)],
        capture_output=True, timeout=180, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.replace(str(globals().get("GITHUB_TOKEN", "\0")), "***")[:400])
    cloned = True
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=str(REPO_DIR), capture_output=True)
    print("cloned", REPO_URL)
except Exception as exc:
    print(f"clone failed ({exc}); trying attached dataset")

if not cloned:
    attached = find_attached_repo()
    if attached is None:
        raise SystemExit("Repo not found. Make REPO_URL reachable or attach it as a Dataset.")
    shutil.copytree(attached, REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))
!pip install -q peft bitsandbytes accelerate sqlglot

# Kaggle preinstalls torchao 0.10.0. PEFT's LoRA dispatcher calls is_torchao_available(),
# which RAISES on a version below its minimum rather than skipping -- which aborts adapter
# injection (ImportError deep inside get_peft_model). We never use torchao (it is an
# optional quantization backend, unrelated to our NF4/bitsandbytes path), so remove it and
# let the check return False cleanly.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
import importlib

importlib.invalidate_caches()
for _mod in list(sys.modules):
    if _mod == "torchao" or _mod.startswith("torchao."):
        del sys.modules[_mod]
try:
    import peft.import_utils as _iu

    if hasattr(_iu.is_torchao_available, "cache_clear"):
        _iu.is_torchao_available.cache_clear()
    print("torchao available to peft:", _iu.is_torchao_available())  # expect False
except Exception as _e:
    print("torchao check note:", _e)

print("repo ready at", REPO_DIR)

In [ ]:
# Download Spider (pairs + databases) and verify the harness BEFORE training.
os.environ["T2S_DATA_DIR"] = "/kaggle/working/data"
os.environ["T2S_ARTIFACTS_DIR"] = f"/kaggle/working/artifacts/experiments/{ARM}"

subprocess.run([sys.executable, str(REPO_DIR / "scripts" / "download_data.py")], check=True, cwd=str(REPO_DIR))
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "check_harness.py"), "--limit", "200"],
    check=True, cwd=str(REPO_DIR),
)

In [ ]:
import importlib
from dataclasses import replace

import text2sql_lora.config as t2s_config
import text2sql_lora.train.sft as sft

importlib.reload(t2s_config)
importlib.reload(sft)

# The ONLY difference between arms: the base precision. Everything else is the default,
# i.e. identical to the published run.
TRAIN_CONFIG = replace(t2s_config.TrainConfig(), load_in_4bit=LOAD_IN_4BIT)
PATHS = t2s_config.Paths()  # picks up T2S_ARTIFACTS_DIR set above

print("arm        :", ARM)
print("load_in_4bit:", TRAIN_CONFIG.load_in_4bit)
print("artifacts  :", PATHS.artifacts_dir)
print("everything else identical to the published run (seed 42, r=16, 3 epochs, batch 16)")

In [ ]:
# SMOKE: 5 steps, 8 dev examples. Surfaces OOM/dtype/shape errors in ~2 min. The fp16
# arm is the important one to smoke -- it takes the newly-guarded non-quantized path.
smoke = sft.run_smoke(
    device="cuda",
    paths=t2s_config.Paths(artifacts_dir=Path("/kaggle/working/smoke")),
)
print("\nsmoke OK. Ignore the accuracy.")

In [ ]:
import gc

del smoke
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# THE REAL ARM. Trains, evaluates all three systems on the full dev set, and writes
# artifacts/experiments/<ARM>/{metrics.json, predictions.jsonl, adapter/}. The manifest
# records peak VRAM and quantization, so the arm is self-describing.
payload = sft.run(
    device="cuda",
    paths=PATHS,
    train_config=TRAIN_CONFIG,
    generation_batch_size=16,
)

In [ ]:
# This arm's numbers, and -- if the other arm's artifact is present -- the side by side.
m = payload["manifest"]
h = payload["headline"]
print(f"ARM {ARM}")
print(f"  base  EX : {h['base_zero_shot_execution_accuracy']:.4f}")
print(f"  tuned EX : {h['lora_zero_shot_execution_accuracy']:.4f}  (+{h['absolute_gain']:.4f})")
print(f"  quant    : {m['quantization']}")
print(f"  wall     : {m['wall_clock_seconds'] / 3600:.2f} h")
print(f"  peak VRAM: {m.get('peak_vram_gb', float('nan')):.2f} GB")

# Merge whatever arms exist (including the published QLoRA run) and print the table.
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "collect_experiments.py")],
    cwd=str(REPO_DIR),
    env={**os.environ, "T2S_ARTIFACTS_DIR": "/kaggle/working/artifacts"},
)

In [ ]:
# Package THIS arm for download. Unzip locally into artifacts/experiments/<ARM>/, then
# run scripts/collect_experiments.py + scripts/report.py to fold it into the README.
shutil.rmtree(PATHS.artifacts_dir / "training", ignore_errors=True)
archive = shutil.make_archive(f"/kaggle/working/{ARM}", "zip", PATHS.artifacts_dir)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)")
for item in sorted(PATHS.artifacts_dir.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(PATHS.artifacts_dir)}  {item.stat().st_size / 1e6:.2f} MB")